In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType

In [0]:
catalog_name = "ecommerce"

###Reading Data from slv_order_items from silver layer      

In [0]:
df = spark.table(f"{catalog_name}.silver.slv_order_items")

df.limit(10).display()

###Adding Gross Amount

In [0]:
df = df.withColumn(
    "gross_amount", F.col("quantity") * F.col("unit_price")
)

###Add discount_amount -> (gross_amount * discount_pct)/ 100

In [0]:
df = df.withColumn(
    "discoun_amount",
    F.ceil(F.col("gross_amount")*(F.col("discount_pct")/100))
    )

###Add sale_amount = gross_amount - discount_amount

In [0]:
df = df.withColumn(
    "sale_amount",
    F.col("gross_amount") - F.col("discoun_amount") + F.col("tax_amount")
    )


###Adding date_id

In [0]:
# Creating Date Key
df = df.withColumn("date_id", F.date_format(F.col("dt"),"yyyyMMdd").cast(IntegerType()))

###Cupon Flag

In [0]:
#For cupon flag, use 1 if cupon_code is not null, esle 0

df = df.withColumn(
    "cupon_flag",
    F.when(F.col("cupon_code").isNotNull(),F.lit(1))\
        .otherwise(F.lit(0))
)


In [0]:
df.limit(5).display()

###Currency Conversion

In [0]:
###Defining fix rates

fix_rates = {

    "INR":1.00,
    "AED":24.18,
    "AUD":57.55,
    "CAD": 62.93,
    "GBP":117.98,
    "SGD": 68.18,
    "USD" : 88.29,
}

#rates = [(k,float(v)) for k,v in fix_rates.items()]
rates =[]
for k, v in fix_rates.items():
    rates.append((k, float(v)))
rates_df = spark.createDataFrame(rates,["currency", "inr_rate"])
rates_df.show()